In [1]:
!pip install -q kaggle

In [4]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API Token: ")

Enter your Kaggle API Token: ··········


In [5]:
!kaggle competitions list

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [6]:
!pip install --upgrade -q kaggle

In [7]:
!kaggle --version

Kaggle CLI 2.2.5


In [8]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API Token: ")

Enter your Kaggle API Token: ··········


In [9]:
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  userRank  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  --------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False         0  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False         0  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False         0  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False         0  
https://www.kaggle.com/compe

In [10]:
!mkdir -p data
!kaggle competitions download -c titanic -p data

100% 34.1k/34.1k [00:00<00:00, 8.57MB/s]



In [11]:
import zipfile

with zipfile.ZipFile("data/titanic.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [12]:
import os

print(os.listdir("data"))

['test.csv', 'titanic.zip', 'train.csv', 'gender_submission.csv']


In [13]:
import pandas as pd

df = pd.read_csv("data/train.csv")

display(df.head())
print("Dataset shape:", df.shape)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Dataset shape: (891, 12)


In [14]:
# Check missing values
missing_values = df.isnull().sum()

print("Missing values in each column:")
display(missing_values)

Missing values in each column:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [15]:
# Separate features and target

X = df.drop(
    columns=["PassengerId", "Name", "Ticket", "Cabin", "Survived"]
)

y = df["Survived"]

print("Features:")
display(X.head())

print("Target:")
display(y.head())

Features:


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


Target:


,Survived
0,0
1,1
2,1
3,1
4,0


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (712, 7)
Testing data: (179, 7)


In [17]:
# Identify numerical and categorical features

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical features: ['Sex', 'Embarked']


In [18]:
from sklearn.impute import SimpleImputer

# Handle missing numerical values
numeric_imputer = SimpleImputer(strategy="median")

# Handle missing categorical values
categorical_imputer = SimpleImputer(strategy="most_frequent")

X_train_numeric = numeric_imputer.fit_transform(
    X_train[numeric_features]
)

X_train_categorical = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

print("Missing numerical values handled.")
print("Missing categorical values handled.")

Missing numerical values handled.
Missing categorical values handled.


In [19]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train_categorical
)

print("Encoded categorical data:")
print(X_train_encoded[:5])

print("Encoded shape:", X_train_encoded.shape)

Encoded categorical data:
[[0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]]
Encoded shape: (712, 5)


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_numeric
)

print("Scaled numerical data:")
print(X_train_scaled[:5])

Scaled numerical data:
[[ 0.82956755 -0.08113533 -0.46508428 -0.46618317  0.5138115 ]
 [-0.37094484 -0.08113533 -0.46508428 -0.46618317 -0.66256323]
 [-1.57145722 -0.08113533 -0.46508428 -0.46618317  3.95539858]
 [ 0.82956755 -0.88782719 -0.46508428  0.72778236 -0.46787435]
 [-0.37094484  0.11093416  0.47833454  0.72778236 -0.11597681]]


In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data after preprocessing:", X_train_processed.shape)
print("Testing data after preprocessing:", X_test_processed.shape)

Training data after preprocessing: (712, 10)
Testing data after preprocessing: (179, 10)


In [22]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# MinMaxScaler
minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X_train_numeric)

print("MinMaxScaler result:")
print(X_minmax[:5])

# RobustScaler
robust_scaler = RobustScaler()
X_robust = robust_scaler.fit_transform(X_train_numeric)

print("\nRobustScaler result:")
print(X_robust[:5])

MinMaxScaler result:
[[1.         0.35285248 0.         0.         0.11027246]
 [0.5        0.35285248 0.         0.         0.        ]
 [0.         0.35285248 0.         0.         0.43288417]
 [1.         0.22090978 0.         0.16666667 0.01824998]
 [0.5        0.3842674  0.125      0.16666667 0.05123659]]

RobustScaler result:
[[ 0.          0.          0.          0.          1.81965184]
 [-1.          0.          0.          0.         -0.6256092 ]
 [-2.          0.          0.          0.          8.97347668]
 [ 0.         -0.75        0.          1.         -0.22092087]
 [-1.          0.17857143  1.          1.          0.51054787]]


In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y_train)

print("Original classes:", label_encoder.classes_)
print("Encoded target:", y_encoded[:10])

Original classes: [0 1]
Encoded target: [1 0 0 1 1 0 1 0 1 0]


In [24]:
print("FINAL PREPROCESSING REPORT")
print("--------------------------")

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

print("Missing values after preprocessing:",
      int(__import__("numpy").isnan(X_train_processed).sum()))

print("Preprocessing completed successfully!")

FINAL PREPROCESSING REPORT
--------------------------
Original training shape: (712, 7)
Processed training shape: (712, 10)
Original testing shape: (179, 7)
Processed testing shape: (179, 10)
Missing values after preprocessing: 0
Preprocessing completed successfully!


## Conclusion

In this lab, I used the Titanic dataset from Kaggle to practice data preprocessing with Scikit-Learn.

The following preprocessing methods were applied:

1. **SimpleImputer** – handled missing values using median and most frequent strategies.
2. **OneHotEncoder** – converted categorical variables into numerical features.
3. **StandardScaler** – standardized numerical features.
4. **MinMaxScaler** – scaled numerical values to a range between 0 and 1.
5. **RobustScaler** – scaled numerical data using the median and interquartile range.
6. **LabelEncoder** – encoded target labels.

I also used train-test splitting, Pipeline, and ColumnTransformer to organize the preprocessing workflow and prevent data leakage.

The final dataset contained 10 processed features. All preprocessing steps were completed successfully.

**Dataset source:** Kaggle – Titanic: Machine Learning from Disaster.